# Scraping Scripts

#### <font color='red'>Please note that the scraping scripts in the file are for academic purposes only</font>

Import Libraries

In [ ]:
#Please pip install the below libraries from terminal before running.

import asyncio
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError
import nest_asyncio
import os
from datetime import datetime, timedelta
import hashlib
import praw
import spacy
from spacy.matcher import PhraseMatcher
from fuzzywuzzy import fuzz
import csv
import time

In [3]:
filepath = r'C:\Users\singh\Downloads\Reddit Files\Comments\crime_related_posts.csv'

Twitter Script

In [ ]:
'''
Below Script Scrapes data from Twitter using PLaywright. Please refer to the playwright installation guide within the ReadMe file for steps to install playwright 
Please refer to the ReadMe file for the relevant changes before running this script
'''

nest_asyncio.apply()


CRIME_PHRASES = [
    "armed robbery", "home invasion", "car theft", "drug trafficking",
    "assault and battery", "domestic violence", "sexual assault",
    "homicide investigation", "gang activity", "cybercrime report",
    "fraud alert", "missing person case", "kidnapping incident",
    "burglary in progress", "shoplifting arrest", "vandalism spree",
    "police chase", "shooting reported", "stabbing victim",
    "crime scene investigation", "murder suspect", "drug bust",
    "human trafficking", "identity theft", "money laundering",
    "terrorist attack", "hate crime", "child abuse", "illegal weapons",
    "carjacking incident", "arson investigation", "bank robbery",
    "prison break", "police brutality", "gang-related violence",
    "cyber attack", "embezzlement scheme", "counterfeit operation",
    "drug overdose", "serial killer", "mass shooting", "bomb threat",
    "hostage situation", "illegal gambling", "organized crime",
    "police officer shot", "wanted fugitive", "crime ring busted"
]
login = "USERNAME"
password = "PASSWORD"

USA_COORDINATES = "39.8283,-98.5795"
SEARCH_RADIUS = "3000mi"

async def login_to_twitter(page, username, password):
    await page.goto("https://twitter.com/login")
    await page.fill('input[autocomplete="username"]', username)
    await page.click('text="Next"')
    await page.fill('input[autocomplete="current-password"]', password)
    await page.click('text="Log in"')
    await page.wait_for_selector('a[aria-label="Profile"]', timeout=30000)

def get_tweet_hash(tweet):
    tweet_text = tweet.get('legacy', {}).get('full_text', '')
    tweet_id = tweet.get('legacy', {}).get('id_str', '')
    return hashlib.md5(f"{tweet_id}:{tweet_text}".encode()).hexdigest()

def parse_tweet_time(time_string):
    return datetime.strptime(time_string, '%a %b %d %H:%M:%S +0000 %Y')

async def scrape_tweets(num_tweets: int = 10000, batch_size: int = 100) -> None:
    _xhr_calls = []
    total_scraped = 0
    batch_number = 1
    phrase_index = 0
    seen_tweets = set()
    phrase_end_times = {phrase: datetime.utcnow() for phrase in CRIME_PHRASES}

    async def intercept_response(response):
        if response.request.resource_type == "xhr" and "SearchTimeline" in response.url:
            _xhr_calls.append(response)

    async with async_playwright() as pw:
        browser = await pw.firefox.launch(headless=False)

        if os.path.exists("twitter_auth.json"):
            context = await browser.new_context(storage_state="twitter_auth.json")
        else:
            context = await browser.new_context()
            page = await context.new_page()
            await login_to_twitter(page, login, password)
            await context.storage_state(path="twitter_auth.json")

        page = await context.new_page()

        page.on("response", intercept_response)

        try:
            while total_scraped < num_tweets:
                current_phrase = CRIME_PHRASES[phrase_index]
                end_time = phrase_end_times[current_phrase]
                tweets = []
                end_time_str = end_time.strftime('%Y-%m-%d_%H:%M:%S_UTC')
                search_query = f'"{current_phrase}" geocode:{USA_COORDINATES},{SEARCH_RADIUS} until:{end_time_str}'
                url = f"https://twitter.com/search?q={search_query}&src=typed_query&f=live"
                await page.goto(url, timeout=60000)  # 60s timeout
                try:
                    await page.wait_for_selector("[data-testid='tweet']", timeout=60000)  # 60s timeout
                except PlaywrightTimeoutError:
                    print(f"No tweets found for '{current_phrase}' before {end_time}. Moving to next phrase.")
                    phrase_index = (phrase_index + 1) % len(CRIME_PHRASES)
                    continue
                scroll_attempts = 0
                max_scroll_attempts = 20
                while len(tweets) < batch_size and scroll_attempts < max_scroll_attempts:
                    await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
                    await asyncio.sleep(2)
                    for xhr in _xhr_calls:
                        try:
                            data = await xhr.json()
                            tweet_entries = data.get('data', {}).get('search_by_raw_query', {}).get('search_timeline', {}).get('timeline', {}).get('instructions', [])
                            for entry in tweet_entries:
                                if entry.get('type') == 'TimelineAddEntries':
                                    for tweet_entry in entry.get('entries', []):
                                        tweet_content = tweet_entry.get('content', {}).get('itemContent', {}).get('tweet_results', {}).get('result', {})
                                        if tweet_content:
                                            tweet_hash = get_tweet_hash(tweet_content)
                                            if tweet_hash not in seen_tweets:
                                                tweet_text = tweet_content.get('legacy', {}).get('full_text', '').lower()

                                                if current_phrase.lower() in tweet_text:
                                                    tweets.append(tweet_content)
                                                    seen_tweets.add(tweet_hash)
                                                    if len(tweets) >= batch_size:
                                                        break
                            if len(tweets) >= batch_size:
                                break
                        except Exception as e:
                            print(f"Error processing xhr response: {e}")
                    _xhr_calls.clear()
                    if len(tweets) >= batch_size:
                        break
                    scroll_attempts += 1
                    await asyncio.sleep(5)
                if tweets:
                    save_tweets_to_csv(tweets, batch_number, current_phrase)
                    total_scraped += len(tweets)

                    last_tweet_time = parse_tweet_time(tweets[-1].get('legacy', {}).get('created_at', ''))
                    phrase_end_times[current_phrase] = last_tweet_time - timedelta(seconds=1)  # Subtract 1 second to avoid duplicates

                    batch_number += 1
                    print(f"Saved batch {batch_number - 1} for phrase '{current_phrase}'. Total tweets scraped: {total_scraped}")
                    print(f"Next batch for '{current_phrase}' will start from: {phrase_end_times[current_phrase]}")
                else:
                    print(f"No tweets found for '{current_phrase}' before {end_time}. Moving to next phrase.")
                phrase_index = (phrase_index + 1) % len(CRIME_PHRASES)
                if phrase_index == 0:
                    print("Completed a full cycle of phrases. Waiting for 5 minutes before starting the next cycle.")
                    await asyncio.sleep(300)  # Wait for 5 minutes
        except Exception as e:
            print(f"An error occurred: {e}")
        finally:
            await browser.close()

def save_tweets_to_csv(tweets: list, batch_number: int, phrase: str):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"Feed_crime_tweets_USA_{phrase.replace(' ', '_')}_batch{batch_number}_{timestamp}.csv"

    with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['tweet_id', 'created_at', 'full_text', 'user_name', 'user_screen_name', 'retweet_count', 'favorite_count', 'location']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

        writer.writeheader()
        for tweet in tweets:
            legacy = tweet.get('legacy', {})
            user = tweet.get('core', {}).get('user_results', {}).get('result', {}).get('legacy', {})
            writer.writerow({
                'tweet_id': legacy.get('id_str'),
                'created_at': legacy.get('created_at'),
                'full_text': legacy.get('full_text'),
                'user_name': user.get('name'),
                'user_screen_name': user.get('screen_name'),
                'retweet_count': legacy.get('retweet_count'),
                'favorite_count': legacy.get('favorite_count'),
                'location': user.get('location')
            })
    print(f"Batch {batch_number} for phrase '{phrase}' saved to {filename}")

async def main():
    await scrape_tweets(num_tweets=10000, batch_size=100)
    print("Scraping completed.")

if __name__ == "__main__":
    asyncio.get_event_loop().run_until_complete(main())

Reddit Script

In [ ]:
'''
Below Script scrapes data from reddit using the free Reddit API. 
Please refer to the ReadMe file for the relevant changes before running this script
'''


nlp = spacy.load("en_core_web_sm")

# Configure Reddit API
reddit = praw.Reddit(
    client_id='pIKrGoabL1M5DtYQJpkjUg', # Replace with your client ID
    client_secret='x_a7Gj1j0c6G9RJAVNRXJvrR1PJBdQ',  # Replace with your client secret
    user_agent='Webscraper for Crime Detection and Prevention by Nearby_Term_1974' # Replace with your user agent
)


CRIME_PHRASES = [
    "crime", "unsafe", "danger", "violent", "theft",
    "robbery", "assault", "suspicious activity", 
    "vandalism", "shooting", "murder", "burglary", 
    "police report", "public safety", "neighborhood safety",
    "breaking and entering", "armed robbery", "crime wave"
]

subreddits_with_locations = [
    ("Buffalo", "Buffalo, NY"),
    ("UpstateNewYork", "Upstate New York"),
    ("BuffaloCrime", "Buffalo, NY"),
    ("UniversityatBuffalo", "Buffalo, NY"),
    ("UBReddit", "Buffalo, NY"),
    ("SouthBuffaloNY", "Buffalo, NY"),
    ("BuffaloNewYork", "Buffalo, NY"),
    ("newyork", "NYC, NY"),
    ("LiveFromNewYork", "NYC, NY"),
    ("nyc", "NYC, NY"),
    ("newyorkcity", "NYC, NY"),
    ("AskNYC", "NYC, NY"),
    ("NYCrimeWatch", "NYC, NY"),
]

matcher = PhraseMatcher(nlp.vocab)
crime_patterns = [nlp.make_doc(phrase) for phrase in CRIME_PHRASES]
matcher.add("CrimePhrases", crime_patterns)

def extract_crime_phrases(text):
    doc = nlp(text)
    matches = matcher(doc)
    return [doc[start:end].text for _, start, end in matches]

def extract_locations(text):
    doc = nlp(text)
    locations = [ent.text for ent in doc.ents if ent.label_ == "GPE"]
    return locations

def scrape_reddit_paginated(subreddit, total_limit=10000, batch_size=10):
    posts = []
    before = None

    while len(posts) < total_limit:
        try:
            subreddit_obj = reddit.subreddit(subreddit)
            if before:
                subreddit_posts = subreddit_obj.new(limit=batch_size, params={"before": int(before)})
            else:
                subreddit_posts = subreddit_obj.new(limit=batch_size)

            batch = [
                {
                    "title": post.title,
                    "body": post.selftext,
                    "score": post.score,
                    "id": post.id,
                    "url": post.url,
                    "created": post.created_utc,
                    "num_comments": post.num_comments,
                    "author": str(post.author),
                    "subreddit": subreddit,
                }
                for post in subreddit_posts
            ]

            if not batch:
                print(f"No more posts found for subreddit: {subreddit}")
                break

            posts.extend(batch)
            before = batch[-1]["created"]
            print(f"Fetched {len(posts)} posts so far from subreddit: {subreddit}")

            if len(batch) < batch_size:
                break

        except Exception as e:
            print(f"An error occurred while fetching posts from {subreddit}: {e}")
            break

    return posts[:total_limit]

def filter_crime_posts(posts, location, threshold=70):
    filtered_posts = []
    for post in posts:
        content = f"{post['title']} {post.get('body', '')}"
        matches = extract_crime_phrases(content)
        fuzzy_score = fuzz.partial_ratio(content.lower(), "crime")
        
        if matches or fuzzy_score >= threshold:
            post['matches'] = matches or ["fuzzy crime match"]
            post['location'] = location
            post['locations_detected'] = extract_locations(content)
            filtered_posts.append(post)
    return filtered_posts

def save_to_csv(data, filepath):
    if not data:
        print("No data to save.")
        return
    keys = data[0].keys()
    with open(filepath, 'w', newline='', encoding='utf-8') as output_file:
        dict_writer = csv.DictWriter(output_file, fieldnames=keys)
        dict_writer.writeheader()
        dict_writer.writerows(data)
    print(f"File saved at: {filepath}")

if __name__ == "__main__":
    total_crime_posts = []
    total_limit = 100000  #Total number of posts to retrieve per subreddit
    batch_size = 10  ####Please do not change, data will get repeated

    for subreddit, location in subreddits_with_locations:
        print(f"Scraping subreddit: {subreddit} (Location: {location})")
        posts = scrape_reddit_paginated(subreddit, total_limit=total_limit, batch_size=batch_size)
        filtered_posts = filter_crime_posts(posts, location)
        total_crime_posts.extend(filtered_posts)
        time.sleep(10)
        
    save_to_csv(total_crime_posts, filepath)